# UBC Pipeline Calibration

Совместный подбор порогов **zone → find → class → merge** на UBC val.

## Перед запуском

Данные: `data/processed_ubc/`, тайлы: `data/raw/ubc/`.
Модели: `convnext_best.pth`, `inria_building_ubc.pth` (или `inria_building_segmenter.pth`), `ny_building_ubc.pth` (или `ny_building_classifier.pth`).

**Test (153 тайла)** — только финальная оценка, без подбора.


In [ ]:
import os
from pathlib import Path

_env_path = Path('..').resolve() / '.env'
if _env_path.exists():
    for _line in _env_path.read_text(encoding='utf-8').splitlines():
        _line = _line.strip()
        if not _line or _line.startswith('#') or '=' not in _line:
            continue
        _key, _val = _line.split('=', 1)
        os.environ.setdefault(_key.strip(), _val.strip().strip('"\''))

_hf_token = os.environ.get('HF_TOKEN')
if _hf_token:
    from huggingface_hub import login
    login(token=_hf_token, add_to_git_credential=False)
    print('Hugging Face: авторизация OK')
else:
    print('HF_TOKEN не найден в .env — веса timm скачаются без токена (медленнее)')


import sys
from pathlib import Path

import torch

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from utils import NUM_WORKERS

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LOADER_WORKERS = NUM_WORKERS
print('PROJECT_ROOT:', PROJECT_ROOT)
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('device:', device, '| DataLoader workers:', LOADER_WORKERS)

import pandas as pd
from utils import INRIA_SEGMENTATION, NY_BUILDING

SEG_BATCH = INRIA_SEGMENTATION.batch_size
CLS_BATCH = NY_BUILDING.batch_size
print('seg batch:', SEG_BATCH, '| cls batch:', CLS_BATCH)



## Модели


In [ ]:
from model_convnext import build_convnext_tiny
from model_segmentation import build_convnext_segmenter
from utils import MODELS_DIR, NY_BUILDING_CLASSES, ZONE_CLASSES

def _pick(primary: str, fallback: str) -> Path:
    p, f = MODELS_DIR / primary, MODELS_DIR / fallback
    if p.exists():
        return p
    if f.exists():
        print(f'  fallback: {fallback}')
        return f
    raise FileNotFoundError(f'Нет {primary} и {fallback}')

zone_ckpt = MODELS_DIR / 'convnext_best.pth'
find_ckpt = _pick('inria_building_ubc.pth', 'inria_building_segmenter.pth')
class_ckpt = _pick('ny_building_ubc.pth', 'ny_building_classifier.pth')

zone_model = build_convnext_tiny(num_classes=len(ZONE_CLASSES), freeze_backbone=False)
zone_model.load_state_dict(torch.load(zone_ckpt, map_location=device, weights_only=True))
zone_model.to(device).eval()

find_model = build_convnext_segmenter(pretrained=False, freeze_encoder=False)
find_model.load_state_dict(torch.load(find_ckpt, map_location=device, weights_only=True))
find_model.to(device).eval()

class_model = build_convnext_tiny(num_classes=len(NY_BUILDING_CLASSES), freeze_backbone=False)
class_model.load_state_dict(torch.load(class_ckpt, map_location=device, weights_only=True))
class_model.to(device).eval()

print('zone:', zone_ckpt.name)
print('find:', find_ckpt.name)
print('class:', class_ckpt.name)


model.safetensors: reconstructing file:   0%|          |  0.00B /  114MB            

model.safetensors: downloading bytes:           |  0.00B            

zone: convnext_best.pth
find: inria_building_ubc.pth
class: ny_building_ubc.pth


## Val-тайлы для калибровки


In [ ]:
from pipeline_ubc import ubc_tile_names_for_eval_split

CALIB_TILE_LIMIT = 80   # None = все 516 val (дольше)
GRID_MODE = 'fast'     # 'fast' быстрее; 'full' — полная сетка

val_tiles = ubc_tile_names_for_eval_split('val')
if CALIB_TILE_LIMIT:
    val_tiles = val_tiles[:CALIB_TILE_LIMIT]
test_tiles = ubc_tile_names_for_eval_split('test')
print(f'Калибровка: {len(val_tiles)} val-тайлов')
print(f'Test eval:  {len(test_tiles)} тайлов')



Калибровка: 80 val-тайлов
Test eval:  153 тайлов


## Кэш zone + find (один раз на тайл)


In [ ]:
from pipeline_calibrate import cache_tile_outputs

caches = []
for tile in tqdm(val_tiles, desc='cache val'):
    caches.append(cache_tile_outputs(
        tile,
        zone_model=zone_model,
        find_model=find_model,
        device=str(device),
        seg_batch_size=SEG_BATCH,
    ))
if not caches:
    raise RuntimeError('Кэш пуст — проверьте zip (тайлы, split.csv, модели)')
print('Кэш val:', len(caches))


cache val:   0%|          | 0/80 [00:00<?, ?it/s]

Кэш val: 80


## Калибровка (fast grid)

Classify вызывается **только** для уникальных пар `(mask_threshold, min_area)`.
Пороги confidence и zone — мгновенная фильтрация/merge.


In [ ]:
import time

from pipeline_calibrate import PipelineParams, calibrate

t0 = time.perf_counter()
result = calibrate(
    caches,
    base_params=PipelineParams(),
    class_model=class_model,
    device=str(device),
    refine=True,
    fast=True,
    grid_mode=GRID_MODE,
    class_batch_size=CLS_BATCH,
)
elapsed = time.perf_counter() - t0

print(f'Калибровка за {elapsed/60:.1f} мин')
print('=== Лучшие параметры ===')
for k, v in result.best_params.to_dict().items():
    print(f'  {k}: {v}')
print()
print('=== Метрики val ===')
for k, v in result.best_metrics.__dict__.items():
    print(f'  {k}: {v}')

import pandas as pd
display(pd.DataFrame(result.top_results).head(10))
if result.best_metrics.n_tiles == 0 or result.best_metrics.joint_score == 0:
    print('⚠ joint=0 или n_tiles=0 — кэш/данные не загрузились, параметры НЕ валидны')


[calib] precompute classify: 9 пар (mask, area) × 80 тайлов


precompute find+class:   0%|          | 0/9 [00:00<?, ?it/s]

[calib]   classify mask=0.40 area=24


[calib]   classify mask=0.40 area=32


[calib]   classify mask=0.40 area=48


...

[calib]   score_weight_mask: 0.4


[calib]   score_weight_class: 0.5


[calib]   score_weight_merge: 0.1


[calib]   joint=0.4321 mask_iou=0.4739 F1=0.2851


Калибровка за 12.0 мин
=== Лучшие параметры ===
  mask_threshold: 0.5
  min_component_area: 40
  class_confidence_threshold: 0.0
  zone_residential_min_prob: 0.0
  match_iou: 0.2
  score_weight_mask: 0.4
  score_weight_class: 0.5
  score_weight_merge: 0.1

=== Метрики val ===
  mask_iou: 0.4739129966549063
  building_macro_f1: 0.28507608802466455
  building_acc: 0.5337629178575818
  resolved_residential_rate: 1.0
  joint_score: 0.4321032426742948
  n_tiles: 80
  n_matched: 1333.0


## Сохранение результатов


In [ ]:
from pipeline_calibrate import DEFAULT_PARAMS_PATH
from utils import REPORTS_DIR, ensure_dirs

ensure_dirs()
params_path = result.best_params.save_json(DEFAULT_PARAMS_PATH)
print('Локально:', params_path)




## Финальная оценка на TEST


In [ ]:
from pipeline_calibrate import evaluate_params, run_pipeline_from_cache
from pipeline_ubc import evaluate_building_classification

best = result.best_params

test_caches = []
for tile in tqdm(test_tiles, desc='cache test'):
    test_caches.append(cache_tile_outputs(
        tile,
        zone_model=zone_model,
        find_model=find_model,
        device=str(device),
        seg_batch_size=SEG_BATCH,
    ))

test_metrics = evaluate_params(test_caches, best, class_model, str(device))
print('=== TEST (frozen params) ===')
for k, v in test_metrics.__dict__.items():
    print(f'  {k}: {v}')

rows = []
for cache in test_caches:
    r = run_pipeline_from_cache(cache, best, class_model, str(device))
    cls = evaluate_building_classification(r, iou_threshold=best.match_iou)
    rows.append({
        'tile': cache.tile_name,
        'mask_iou': r.mask_iou,
        'building_macro_f1': cls['macro_f1'],
        'building_acc': cls['accuracy'],
        'matched': cls['matched'],
    })
test_df = pd.DataFrame(rows)
test_csv = REPORTS_DIR / 'pipeline_calibrated_test_metrics.csv'
test_df.to_csv(test_csv, index=False)
print('CSV:', test_csv)


display(test_df.head(10))



cache test:   0%|          | 0/153 [00:00<?, ?it/s]

=== TEST (frozen params) ===
  mask_iou: 0.5253233366102366
  building_macro_f1: 0.26241387080121925
  building_acc: 0.46966493134479215
  resolved_residential_rate: 0.9934640522875817
  joint_score: 0.4406826752734624
  n_tiles: 153
  n_matched: 3227.0
